# The Bridge and the Torch

In the darkness of the night, a group of four individuals encounters a river. A slender bridge stretches before them, capable of accommodating just two people simultaneously. Equipped with a single torch, they must rely on its flickering light to navigate the bridge. Each person possesses a distinct crossing time: Ariela takes 1 minute, Brian takes 2 minutes, Charly takes 5 minutes, and Dumpy takes 8 minutes. It is crucial to note that when two people cross together, they must synchronize their steps with the slower individual's pace. Given the torch's limited lifespan of 15 minutes, the pressing question arises: can all four individuals successfully traverse the bridge?

In [1]:
// load the formatting setup and the style sheet, install Z3 if necessary
#load "../setup.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

We need the following variables to encode the problem:
* `A` equals `1` if Ariela is on the left shore,
* `B` equals `1` if Brian  is on the left shore,
* `C` equals `1` if Charly is on the left shore,
* `D` equals `1` if Dumpy  is on the left shore,
* `T` equals the time that has passed.

These variables are Z3 variables of type `IntExpr`.  Constraints are values of type `BoolExpr`.  The operators `.=`, `.<=`, `.>=`, etc. are defined in `z3.fsx`.

The function `start` takes five Z3 variables `A`, `B`, `C`, `D`, and `T` describing a state.  It returns a list of constraints that specify that everybody is on the left side of the river and that no time has passed.

In [3]:
let start (A: IntExpr) (B: IntExpr) (C: IntExpr) (D: IntExpr) (T: IntExpr) : BoolExpr list =
    [ A .= 1; B .= 1; C .= 1; D .= 1; T .= 0 ]

The function `goal` takes five Z3 variables `A`, `B`, `C`, `D`, and `T` describing a state.  It returns a list of constraints that specify that everybody is on the right side of the river and that at most 15 minutes have passed.

In [4]:
let goal (A: IntExpr) (B: IntExpr) (C: IntExpr) (D: IntExpr) (T: IntExpr) : BoolExpr list =
    [ A .= 0; B .= 0; C .= 0; D .= 0; T .<= 15 ]

The function `torchOK` describes one crossing of the bridge from left to right.  It takes ten arguments:
* `A` is the number of Arielas on the left side before the crossing, 
   while `Ax` is the number of Arielas on the left side after the crossing.
* Dito for `B`, `Bx`, $\cdots$, `D`, `Dx`.
* `T` is the time before the crossing and `Tx` is the time after the crossing.

It returns a list of constraints that specify that one or two persons cross the bridge from left to right and that the time needed for this crossing has passed.

In [5]:
let torchOK (A: IntExpr) (B: IntExpr) (C: IntExpr) (D: IntExpr) (T: IntExpr)
            (Ax: IntExpr) (Bx: IntExpr) (Cx: IntExpr) (Dx: IntExpr) (Tx: IntExpr) : BoolExpr list =
    [ A .>= Ax
      B .>= Bx
      C .>= Cx
      D .>= Dx
      Implies(A .> Ax, Tx .>= T + 1)   // Ariela
      Implies(B .> Bx, Tx .>= T + 2)   // Brian
      Implies(C .> Cx, Tx .>= T + 5)   // Charly
      Implies(D .> Dx, Tx .= T + 8)    // Dumpy
      (A + B + C + D) - (Ax + Bx + Cx + Dx) .<= 2
      (A + B + C + D) - (Ax + Bx + Cx + Dx) .>= 1 ]

The function `transition` returns a list of constraints that describe a crossing of the bridge.  It takes eleven arguments: the variables `A`, $\cdots$, `T` describing the state before the crossing, the variables `Ax`, $\cdots$, `Tx` describing the state after the crossing, and the number `i` of the crossing.  The first crossing has `i = 0`.  If `i` is even, the persons cross from left to right, otherwise they return from right to left.  In the second case, we can use the function `torchOK` with the roles of the states before and after the crossing interchanged.

In [6]:
let transition (A: IntExpr) (B: IntExpr) (C: IntExpr) (D: IntExpr) (T: IntExpr)
               (Ax: IntExpr) (Bx: IntExpr) (Cx: IntExpr) (Dx: IntExpr) (Tx: IntExpr) (i: int) : BoolExpr list =
    if i % 2 = 0 then
        torchOK A  B  C  D  T Ax Bx Cx Dx Tx
    else
        torchOK Ax Bx Cx Dx T A  B  C  D  Tx

The function `bridgeCSP` takes one argument:
- `n` is the number of crossings.

It tries to solve the problem using `n` crossings of the bridge.  If this is possible, `Some Solution` is returned, where `Solution` is a map assigning values to the variables.  Otherwise, `None` is returned.  The variables describing the state after the $i^{\textrm{th}}$ crossing are `A[i]`, `B[i]`, `C[i]`, `D[i]`, and `T[i]`.

In [7]:
let bridgeCSP (n: int) : Map<string, int> option =
    let S = newSolver ()
    let A = [| for i in 0 .. n -> Int (sprintf "A%d" i) |]
    let B = [| for i in 0 .. n -> Int (sprintf "B%d" i) |]
    let C = [| for i in 0 .. n -> Int (sprintf "C%d" i) |]
    let D = [| for i in 0 .. n -> Int (sprintf "D%d" i) |]
    let T = [| for i in 0 .. n -> Int (sprintf "T%d" i) |]
    let Cts = [ yield! start A[0] B[0] C[0] D[0] T[0]
                yield! goal  A[n] B[n] C[n] D[n] T[n]
                for i in 0 .. n - 1 do
                    let j = i + 1
                    yield! transition A[i] B[i] C[i] D[i] T[i] A[j] B[j] C[j] D[j] T[j] i
                    for X in [ A; B; C; D ] do
                        yield 0 .<= X[i]
                        yield X[i] .<= 1
                    yield 0 .<= T[i] ]
    S.Add(Array.ofList Cts)
    if S.Check() = Status.SATISFIABLE then
        let m = S.Model
        Some (Map [ for (name, X) in [ ("A", A); ("B", B); ("C", C); ("D", D); ("T", T) ] do
                      for i in 0 .. n -> (sprintf "%s%d" name i, intValue m X[i]) ])
    else
        None

The function `findSolution` takes no arguments.  It tries $n = 1, 3, 5, \cdots$ crossings until a solution is found.  It returns the number of crossings together with the solution.

In [8]:
let findSolution () : int * Map<string, int> =
    let rec search (n: int) =
        printfn "%d" n
        match bridgeCSP n with
        | Some solution -> (n, solution)
        | None          -> search (n + 2)
    search 1

In [9]:
#!time
let n, Solution = findSolution ()

1

3

5

Wall time: 32.4277ms

In [10]:
n, Solution

(5, map [("A0", 1); ("A1", 0); ("A2", 1); ("A3", 1); ("A4", 1); ("A5", 0); ("B0", 1); ("B1", 0); ("B2", 0); ("B3", 0); ("B4", 1); ("B5", 0); ("C0", 1); ("C1", 1); ("C2", 1); ("C3", 0); ("C4", 0); ("C5", 0); ("D0", 1); ("D1", 1); ("D2", 1); ("D3", 0); ("D4", 0); ("D5", 0); ("T0", 0); ("T1", 2); ("T2", 3); ("T3", 11); ("T4", 13); ("T5", 15)])

The function `showSolution` takes two arguments:
- `Solution` is a map assigning values to the variables,
- `n` is the number of crossings.

It prints the solution.  For every crossing, the persons on both sides of the river, the time that has passed, and the persons crossing the bridge are shown.

In [11]:
let showSolution (Solution: Map<string, int>) (n: int) : unit =
    let rep (k: int) (s: string) = String.replicate k s
    let get name i = Solution[sprintf "%s%d" name i]
    let persons A B C D = rep A "🏃‍♀️" + rep B "🏃🏽‍♂️" + rep C "🚶🏽‍♂️" + rep D "👨‍🦽"
    for i in 0 .. n do
        let A, B, C, D, T = get "A" i, get "B" i, get "C" i, get "D" i, get "T" i
        printfn "%s%s%s" (persons A B C D) (String.replicate 42 " ") (persons (1-A) (1-B) (1-C) (1-D))
        printfn "🕰️%d" T
        if i < n then
            let sign = if i % 2 = 0 then 1 else -1
            let crossing name = sign * (get name i - get name (i+1))
            let arrows = if i % 2 = 0 then ">>" else "<<"
            printfn "%s%s %s %s" (String.replicate 24 " ") arrows
                    (persons (crossing "A") (crossing "B") (crossing "C") (crossing "D")) arrows

In [12]:
showSolution Solution n

🏃‍♀️🏃🏽‍♂️🚶🏽‍♂️👨‍🦽

🕰️

0

>>

🏃‍♀️🏃🏽‍♂️

...

🏃‍♀️🏃🏽‍♂️

>>

🏃‍♀️🏃🏽‍♂️🚶🏽‍♂️👨‍🦽

🕰️

15